In [ ]:
import pandas as pd
import numpy as np

def discretize_column(df_input, col, nbin=10, mode='equiwidth'):
    """
    Funzione per discretizzare una colonna.
    """
    
    # 1. CORREZIONE IMPORTANTE: COPIA VERA
    # Se scrivi df = df_input, stai solo creando un "alias". 
    # Se modifichi df, modifichi anche l'originale fuori dalla funzione.
    # Usa .copy() per lavorare su un binario separato.
    df = df_input.copy()

    # Estraiamo la serie su cui lavorare per comodità
    series = df.iloc[:, col]

    # ==========================
    # MODALITÀ EQUI-WIDTH
    # ==========================
    if mode == 'equiwidth':
        # Calcolo Range
        min_val = series.min()
        max_val = series.max()
        range_val = max_val - min_val

        # Formula di proporzione (Vettorializzata)
        # Nota: Aggiungiamo un piccolo controllo. Se un valore è esattamente uguale al max,
        # la formula darebbe 'nbin' (che è fuori indice). Usiamo .clip per tenerlo dentro.
        
        # Calcolo indici
        bins = ((series - min_val) * nbin // range_val)
        
        # Correggiamo il caso limite: il valore massimo finirebbe nel bin nbin (che non esiste)
        # Lo costringiamo a restare nel bin (nbin-1)
        bins = bins.clip(upper=nbin-1)
        
        # Sostituzione nel dataframe
        df.iloc[:, col] = bins.astype(int)


    # ==========================
    # MODALITÀ EQUI-DEPTH
    # ==========================
    if mode == 'equidepth':
        
        # 1. JITTERING VETTORIALE (Senza ciclo for)
        # Generiamo N numeri casuali in un colpo solo e li sommiamo
        noise = np.random.random(len(series)) * 0.01
        series_noisy = series + noise

        # 2. CALCOLO SOGLIE (Thresholds)
        # Invece di ordinare e contare a mano (lento), usiamo i quantili.
        # Se nbin=10, vogliamo i tagli a 0%, 10%, 20%... 100%
        # np.linspace(0, 1, 11) crea [0.0, 0.1, 0.2 ... 1.0]
        quantiles = np.linspace(0, 1, nbin + 1)
        bins_edges = series_noisy.quantile(quantiles)

        # 3. ASSEGNAZIONE (Senza ciclo for)
        # pd.cut divide la serie usando i bordi (edges) che abbiamo calcolato sopra.
        # labels=False restituisce 0, 1, 2... invece di intervalli tipo (0.5, 1.2]
        # include_lowest=True serve per includere anche il valore minimo nel primo bin
        df.iloc[:, col] = pd.cut(series_noisy, bins=bins_edges, labels=False, include_lowest=True)

    return df